In [1]:
import sqlite3
import pandas as pd

In [4]:
# Load cleaned data back into memory
df_master = pd.read_csv('../data/processed/clean_streaming_activity.csv')

In [3]:
# Create connection from notebook to file-based SQLite database
conn = sqlite3.connect('../data/streaming_warehouse.db')

In [5]:
# Write the master data into a table in the SQLite database
df_master.to_sql('stg_streaming_activity', conn, if_exists='replace', index=False)

18075

In [6]:
# Close the connection safely
conn.close()

In [14]:
# Re-open connection to look inside
conn = sqlite3.connect('../data/streaming_warehouse.db')

In [8]:
# Write SQL query to check first few rows of new table
query = "SELECT * FROM stg_streaming_activity LIMIT 5;"
df_verification = pd.read_sql_query(query, conn)

In [10]:
# View SQL query results
display(df_verification)

,log_id,user_id,activity_date,device_type,minutes_streamed,ad_clicks_avoided,signup_date,plan_type,payment_method,status,cancellation_date
0,LOG_100001,USR_0001,2025-01-01,TV,159,3,2025-01-01,Individual,Google Wallet,Canceled,2025-06-16
1,LOG_100002,USR_0001,2025-01-05,Mobile,16,1,2025-01-01,Individual,Google Wallet,Canceled,2025-06-16
2,LOG_100003,USR_0001,2025-01-09,Mobile,55,1,2025-01-01,Individual,Google Wallet,Canceled,2025-06-16
3,LOG_100004,USR_0001,2025-01-13,TV,12,2,2025-01-01,Individual,Google Wallet,Canceled,2025-06-16
4,LOG_100005,USR_0001,2025-01-17,Mobile,34,3,2025-01-01,Individual,Google Wallet,Canceled,2025-06-16


In [17]:
print(df_verification.columns.tolist())

['log_id', 'user_id', 'activity_date', 'device_type', 'minutes_streamed', 'ad_clicks_avoided', 'signup_date', 'plan_type', 'payment_method', 'status', 'cancellation_date']


In [15]:
# Who are your top streamers?
# Calculate total watch time per user and display results in descending order
top_users_query = """
SELECT 
    user_id,
    SUM(minutes_streamed) AS total_watch_time,
    COUNT(log_id) AS total_sessions
FROM stg_streaming_activity
GROUP BY user_id
ORDER BY total_watch_time DESC
LIMIT 10
"""

df_top_users = pd.read_sql_query(top_users_query, conn)
display(df_top_users)

,user_id,total_watch_time,total_sessions
0,USR_0401,2537,39
1,USR_0867,2505,39
2,USR_0417,2412,38
3,USR_0715,2408,39
4,USR_0187,2374,35
5,USR_0310,2330,39
6,USR_0664,2318,39
7,USR_0151,2317,39
8,USR_0982,2278,40
9,USR_0727,2268,34


In [16]:
# Close connection
conn.close()